READER & SETUP

In [0]:
from pyspark.sql.functions import (
    col, from_json, regexp_replace,regexp_extract, to_date, to_timestamp, explode,
    datediff, floor, current_date, current_timestamp, round, unix_timestamp, element_at
)
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, DoubleType, BooleanType
)


# Read streaming Bronze table once
df_bronze = spark.readStream.table("healthcare_catalog.default.bronze_clinical_events")

**1.PATIENTS DEMOGRAPHIC**

In [0]:
# 1. Patient FHIR Schema
patient_schema = StructType([
    StructField("resourceType", StringType()),
    StructField("id", StringType()),
    StructField("gender", StringType()),
    StructField("birthDate", StringType()),
    StructField("maritalStatus", StructType([StructField("text", StringType())])),
    StructField("address", ArrayType(StructType([
        StructField("line", ArrayType(StringType())),
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("postalCode", StringType())
    ])))
])

# 2. Extract & Flatten
df_patients = (
    df_bronze
    .withColumn("parsed", from_json(col("raw_payload"), patient_schema))
    .filter(col("parsed.resourceType") == "Patient")
    .select(
        col("parsed.id").alias("patient_id"),
        col("parsed.gender").alias("gender"),
        to_date(col("parsed.birthDate")).alias("birth_date"),
        floor(datediff(current_date(), to_date(col("parsed.birthDate"))) / 365.25).alias("age"),
        col("parsed.maritalStatus.text").alias("marital_status"),
        col("parsed.address")[0]["line"][0].alias("street_address"),
        col("parsed.address")[0]["city"].alias("city"),
        col("parsed.address")[0]["state"].alias("state"),
        col("parsed.address")[0]["postalCode"].alias("postal_code"),
        current_timestamp().alias("processed_at")
    )
)

# 3. Stream Write
query_patients = (
    df_patients.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/healthcare_catalog/default/checkpoints/chk_silver_patient2")
    .trigger(availableNow=True)
    .toTable("healthcare_catalog.default.silver_patient_demo_staging")
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# ==========================================
# 1. DEFINE REUSABLE SCD2 MERGE FUNCTION
# ==========================================
def merge_scd2_patients(df_staging, batch_id=None, target_table_name="healthcare_catalog.default.silver_patients"):
    """
    Executes an SCD Type 2 Merge from staging DataFrame into the target Delta table.
    Fully compatible with Databricks Serverless Compute.
    """
    # Deduplicate staging data per patient_id to keep the latest record in this batch
    window_spec = Window.partitionBy("patient_id").orderBy(F.col("processed_at").desc())
    
    stg_dedup = (
        df_staging
        .withColumn("row_num", F.row_number().over(window_spec))
        .filter("row_num = 1")
        .drop("row_num")
    )

    # Reference target Delta table
    target_table = DeltaTable.forName(spark, target_table_name)
    
    # Build merge payload for insert vs update
    stg_updates = stg_dedup.withColumn("merge_key", F.col("patient_id"))
    stg_inserts = stg_dedup.withColumn("merge_key", F.lit(None))
    
    stg_payload = stg_updates.unionByName(stg_inserts)
    
    # Perform atomic Delta MERGE
    (
        target_table.alias("target")
        .merge(
            stg_payload.alias("stg"),
            "target.patient_id = stg.merge_key AND target.is_current = true"
        )
        .whenMatchedUpdate(
            condition="""
                target.gender != stg.gender OR 
                target.marital_status != stg.marital_status OR 
                target.street_address != stg.street_address OR 
                target.city != stg.city OR
                target.state != stg.state OR
                target.postal_code != stg.postal_code
            """,
            set={
                "is_current": F.lit(False),
                "valid_to": F.col("stg.processed_at")
            }
        )
        .whenNotMatchedInsert(
            values={
                "patient_id": "stg.patient_id",
                "gender": "stg.gender",
                "birth_date": "stg.birth_date",
                "age": "stg.age",
                "marital_status": "stg.marital_status",
                "street_address": "stg.street_address",
                "city": "stg.city",
                "state": "stg.state",
                "postal_code": "stg.postal_code",
                "valid_from": "stg.processed_at",
                "valid_to": "NULL",
                "is_current": "true"
            }
        )
        .execute()
    )


# ==========================================
# 2. SERVERLESS-SAFE EXECUTION DRIVER SCRIPT
# ==========================================
STAGING_TABLE = "healthcare_catalog.default.silver_patient_demo_staging"
TARGET_TABLE = "healthcare_catalog.default.silver_patients"

df_staging = spark.read.table(STAGING_TABLE)

# SERVERLESS FIX: df.isEmpty() replaces df.rdd.isEmpty()
if df_staging.isEmpty():
    print(f"No records found in {STAGING_TABLE}. Skipping SCD2 run.")
else:
    # Optional: limit action to head(1) or count for logging
    print(f"Staging records found in {STAGING_TABLE}. Running SCD2 Merge...")
    merge_scd2_patients(df_staging, batch_id=0, target_table_name=TARGET_TABLE)
    print("SCD2 batch merge successfully completed on Serverless compute.")

Staging records found in healthcare_catalog.default.silver_patient_demo_staging. Running SCD2 Merge...
SCD2 batch merge successfully completed on Serverless compute.


In [0]:
%sql
select*
FROM healthcare_catalog.default.silver_patients

patient_id,gender,birth_date,age,marital_status,street_address,city,state,postal_code,valid_from,valid_to,is_current
25cc2755-9643-43a9-837c-f79c04ad8916,female,1997-08-08,28,Never Married,802 Pfannerstill Skyway,Westford,Massachusetts,null,2026-07-26T21:19:50.105Z,null,true
31191928-6acb-4d73-931c-e601cc3a13fa,female,2002-10-24,23,Never Married,892 Hoppe Annex,Wakefield,Massachusetts,01880,2026-07-26T21:19:50.105Z,null,true
346d4b95-5e00-48fe-a21e-076735ca1d74,male,2001-11-29,24,Never Married,113 Abshire Heights,Ludlow,Massachusetts,null,2026-07-26T21:19:50.105Z,null,true
3f8be6b0-15a6-43f4-87f3-1adb737fa598,male,1993-02-17,33,Never Married,506 Goldner Parade,Brookline,Massachusetts,02215,2026-07-26T21:19:50.105Z,null,true
5c818f3d-7051-4b86-8203-1dc624a91804,male,1997-12-26,28,Never Married,441 Zemlak Union Unit 91,Milton,Massachusetts,02186,2026-07-26T21:19:50.105Z,null,true
5cbc121b-cd71-4428-b8b7-31e53eba8184,male,1945-12-10,80,S,894 Brakus Bypass,Taunton,Massachusetts,02718,2026-07-26T21:19:50.105Z,null,true
5dc02d13-5c69-4c36-87d5-16738f088300,male,1996-12-04,29,Never Married,217 Will Spur Suite 31,Amherst,Massachusetts,null,2026-07-26T21:19:50.105Z,null,true
668605fe-a8dc-4601-ae48-f5bc24bbea74,female,1999-11-14,26,Never Married,721 Von Mews Unit 51,Plymouth,Massachusetts,02360,2026-07-26T21:19:50.105Z,null,true
67816396-e325-496d-a6ec-c047756b7ce4,male,1999-12-12,26,Never Married,638 Brakus Union Suite 44,Northbridge,Massachusetts,null,2026-07-26T21:19:50.105Z,null,true
6d6fec2a-b149-49c1-b669-4b7106a7aa72,male,1969-12-26,56,M,176 Walsh Course Suite 77,Worcester,Massachusetts,01545,2026-07-26T21:19:50.105Z,null,true


**2.ENCOUNTERS**

In [0]:
# 1. Encounter FHIR Schema
encounter_schema = StructType([
    StructField("resourceType", StringType()),
    StructField("id", StringType()),
    StructField("status", StringType()),
    StructField("subject", StructType([StructField("reference", StringType())])),
    StructField("period", StructType([
        StructField("start", StringType()),
        StructField("end", StringType())
    ])),
    StructField("class", StructType([
        StructField("code", StringType()),
        StructField("display", StringType())
    ]))
])

# 2. Extract & Flatten
df_encounters = (
    df_bronze
    .withColumn("parsed", from_json(col("raw_payload"), encounter_schema))
    .filter(col("parsed.resourceType") == "Encounter")
    .select(
        col("parsed.id").alias("encounter_id"),
        regexp_replace(col("parsed.subject.reference"), "^urn:uuid:", "").alias("patient_id"),
        to_timestamp(col("parsed.period.start")).alias("admission_date"),
        to_timestamp(col("parsed.period.end")).alias("discharge_date"),
        # Calculate Length of Stay (LOS) in days
        round((unix_timestamp(to_timestamp(col("parsed.period.end"))) - 
               unix_timestamp(to_timestamp(col("parsed.period.start")))) / 86400, 2).alias("length_of_stay_days"),
        col("parsed.class.code").alias("admission_type_code"),
        col("parsed.class.display").alias("admission_type"),
        col("parsed.status").alias("discharge_status"),
        current_timestamp().alias("processed_at")
    )
)

# 3. Stream Write
query_encounters = (
    df_encounters.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/healthcare_catalog/default/checkpoints/chk_silver_encounterr")
    .trigger(availableNow=True)
    .toTable("healthcare_catalog.default.silver_encounter_staging")
)

In [0]:
%sql
MERGE INTO healthcare_catalog.default.silver_encounters t
USING (
    SELECT *
    FROM healthcare_catalog.default.silver_encounter_staging
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY encounter_id 
        ORDER BY processed_at DESC
    ) = 1
) s
ON t.encounter_id = s.encounter_id

WHEN MATCHED THEN
  UPDATE SET 
    t.patient_id = s.patient_id,
    t.admission_date = s.admission_date,
    t.discharge_date = s.discharge_date,
    t.length_of_stay_days = s.length_of_stay_days,
    t.admission_type_code = s.admission_type_code,
    t.admission_type = s.admission_type,
    t.discharge_status = s.discharge_status,
    t.processed_at = s.processed_at

WHEN NOT MATCHED THEN
  INSERT (
    encounter_id,
    patient_id,
    admission_date,
    discharge_date,
    length_of_stay_days,
    admission_type_code,
    admission_type,
    discharge_status,
    processed_at
  )
  VALUES (
    s.encounter_id,
    s.patient_id,
    s.admission_date,
    s.discharge_date,
    s.length_of_stay_days,
    s.admission_type_code,
    s.admission_type,
    s.discharge_status,
    s.processed_at
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
923,568,0,355


**PATIENT CONDITIONS**

In [0]:
# 1. Condition FHIR Schema
condition_schema = StructType([
    StructField("resourceType", StringType()),
    StructField("id", StringType()),
    StructField("subject", StructType([StructField("reference", StringType())])),
    StructField("encounter", StructType([StructField("reference", StringType())])),
    StructField("onsetDateTime", StringType()),
    StructField("code", StructType([
        StructField("coding", ArrayType(StructType([
            StructField("code", StringType()),
            StructField("display", StringType())
        ]))),
        StructField("text", StringType())
    ]))
])

# 2. Extract & Flatten
df_conditions = (
    df_bronze
    .withColumn("parsed", from_json(col("raw_payload"), condition_schema))
    .filter(col("parsed.resourceType") == "Condition")
    .select(
        col("parsed.id").alias("condition_id"),
        regexp_replace(col("parsed.subject.reference"), "^urn:uuid:", "").alias("patient_id"),
        regexp_replace(col("parsed.encounter.reference"), "^urn:uuid:", "").alias("encounter_id"),
        col("parsed.code.coding")[0]["code"].alias("diagnosis_code"),
        col("parsed.code.coding")[0]["display"].alias("diagnosis_description"),
        to_date(col("parsed.onsetDateTime")).alias("onset_date"),
        current_timestamp().alias("processed_at")
    )
)

# 3. Stream Write
query_conditions = (
    df_conditions.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/healthcare_catalog/default/checkpoints/chk_silver_conditionn1")
    .trigger(availableNow=True)
    .toTable("healthcare_catalog.default.silver_conditions_staging")
)

In [0]:
%sql
MERGE INTO healthcare_catalog.default.silver_conditions t
USING (
    SELECT *
    FROM healthcare_catalog.default.silver_conditions_staging
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY condition_id 
        ORDER BY processed_at DESC
    ) = 1
) s
ON t.condition_id = s.condition_id

WHEN MATCHED THEN
  UPDATE SET 
    t.patient_id = s.patient_id,
    t.encounter_id = s.encounter_id,
    t.diagnosis_code = s.diagnosis_code,
    t.diagnosis_description = s.diagnosis_description,
    t.onset_date = s.onset_date,
    t.processed_at = s.processed_at

WHEN NOT MATCHED THEN
  INSERT (
    condition_id,
    patient_id,
    encounter_id,
    diagnosis_code,
    diagnosis_description,
    onset_date,
    processed_at
  )
  VALUES (
    s.condition_id,
    s.patient_id,
    s.encounter_id,
    s.diagnosis_code,
    s.diagnosis_description,
    s.onset_date,
    s.processed_at
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
130,130,0,0


CLINICAL OBSERVATIONS

In [0]:
# 1. Observation FHIR Schema
observation_schema = StructType([
    StructField("resourceType", StringType()),
    StructField("id", StringType()),
    StructField("subject", StructType([StructField("reference", StringType())])),
    StructField("encounter", StructType([StructField("reference", StringType())])),
    StructField("effectiveDateTime", StringType()),
    StructField("code", StructType([
        StructField("text", StringType()),
        StructField("coding", ArrayType(StructType([StructField("display", StringType())])))
    ])),
    StructField("valueQuantity", StructType([
        StructField("value", DoubleType()),
        StructField("unit", StringType())
    ])),
    StructField("valueString", StringType())
])

# 2. Extract & Flatten
df_observations = (
    df_bronze
    .withColumn("parsed", from_json(col("raw_payload"), observation_schema))
    .filter(col("parsed.resourceType") == "Observation")
    .select(
        col("parsed.id").alias("observation_id"),
        regexp_replace(col("parsed.subject.reference"), "^urn:uuid:", "").alias("patient_id"),
        regexp_replace(col("parsed.encounter.reference"), "^urn:uuid:", "").alias("encounter_id"),
        col("parsed.code.text").alias("observation_type"),
        col("parsed.valueQuantity.value").alias("result_numeric_value"),
        col("parsed.valueQuantity.unit").alias("unit_of_measure"),
        col("parsed.valueString").alias("result_string_value"),
        to_timestamp(col("parsed.effectiveDateTime")).alias("observation_timestamp"),
        current_timestamp().alias("processed_at")
    )
)

# 3. Stream Write
query_observations = (
    df_observations.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/healthcare_catalog/default/checkpoints/chk_silver_observation2")
    .trigger(availableNow=True)
    .toTable("healthcare_catalog.default.silver_observations_staging")
)

In [0]:
%sql
MERGE INTO healthcare_catalog.default.silver_observations t
USING (
    SELECT *
    FROM healthcare_catalog.default.silver_observations_staging
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY observation_id 
        ORDER BY processed_at DESC
    ) = 1
) s
ON t.observation_id = s.observation_id

WHEN MATCHED THEN
  UPDATE SET 
    t.patient_id = s.patient_id,
    t.encounter_id = s.encounter_id,
    t.observation_type = s.observation_type,
    t.result_numeric_value = s.result_numeric_value,
    t.unit_of_measure = s.unit_of_measure,
    t.result_string_value = s.result_string_value,
    t.observation_timestamp = s.observation_timestamp,
    t.processed_at = s.processed_at

WHEN NOT MATCHED THEN
  INSERT (
    observation_id,
    patient_id,
    encounter_id,
    observation_type,
    result_numeric_value,
    unit_of_measure,
    result_string_value,
    observation_timestamp,
    processed_at
  )
  VALUES (
    s.observation_id,
    s.patient_id,
    s.encounter_id,
    s.observation_type,
    s.result_numeric_value,
    s.unit_of_measure,
    s.result_string_value,
    s.observation_timestamp,
    s.processed_at
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2743,2743,0,0


MEDICATION REQUEST

In [0]:
# 1. MedicationRequest FHIR Schema
medication_schema = StructType([
    StructField("resourceType", StringType()),
    StructField("id", StringType()),
    StructField("status", StringType()),
    StructField("subject", StructType([StructField("reference", StringType())])),
    StructField("encounter", StructType([StructField("reference", StringType())])),
    StructField("authoredOn", StringType()),
    StructField("medicationCodeableConcept", StructType([
        StructField("text", StringType()),
        StructField("coding", ArrayType(StructType([
            StructField("code", StringType()),
            StructField("display", StringType())
        ])))
    ]))
])

# 2. Extract & Flatten
df_medications = (
    df_bronze
    .withColumn("parsed", from_json(col("raw_payload"), medication_schema))
    .filter(col("parsed.resourceType") == "MedicationRequest")
    .select(
        col("parsed.id").alias("medication_request_id"),
        regexp_replace(col("parsed.subject.reference"), "^urn:uuid:", "").alias("patient_id"),
        regexp_replace(col("parsed.encounter.reference"), "^urn:uuid:", "").alias("encounter_id"),
        col("parsed.medicationCodeableConcept.coding")[0]["display"].alias("medication_name"),
        col("parsed.medicationCodeableConcept.coding")[0]["code"].alias("rxnorm_code"),
        col("parsed.status").alias("prescription_status"),
        to_date(col("parsed.authoredOn")).alias("prescribed_date"),
        current_timestamp().alias("processed_at")
    )
)

# 3. Stream Write
query_medications = (
    df_medications.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/healthcare_catalog/default/checkpoints/chk_silver_medications_1")
    .trigger(availableNow=True)
    .toTable("healthcare_catalog.default.silver_medication_requests_staging")
)

In [0]:
%sql
MERGE INTO healthcare_catalog.default.silver_medication_requests t
USING (
    SELECT *
    FROM healthcare_catalog.default.silver_medication_requests_staging
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY medication_request_id 
        ORDER BY processed_at DESC
    ) = 1
) s
ON t.medication_request_id = s.medication_request_id

WHEN MATCHED THEN
  UPDATE SET 
    t.patient_id = s.patient_id,
    t.encounter_id = s.encounter_id,
    t.medication_name = s.medication_name,
    t.rxnorm_code = s.rxnorm_code,
    t.prescription_status = s.prescription_status,
    t.prescribed_date = s.prescribed_date,
    t.processed_at = s.processed_at

WHEN NOT MATCHED THEN
  INSERT (
    medication_request_id,
    patient_id,
    encounter_id,
    medication_name,
    rxnorm_code,
    prescription_status,
    prescribed_date,
    processed_at
  )
  VALUES (
    s.medication_request_id,
    s.patient_id,
    s.encounter_id,
    s.medication_name,
    s.rxnorm_code,
    s.prescription_status,
    s.prescribed_date,
    s.processed_at
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
85,85,0,0


PROCEDURES

In [0]:
# 1. Procedure FHIR Schema
procedure_schema = StructType([
    StructField("resourceType", StringType()),
    StructField("id", StringType()),
    StructField("status", StringType()),
    StructField("subject", StructType([StructField("reference", StringType())])),
    StructField("encounter", StructType([StructField("reference", StringType())])),
    StructField("performedDateTime", StringType()),
    StructField("code", StructType([
        StructField("text", StringType()),
        StructField("coding", ArrayType(StructType([
            StructField("code", StringType()),
            StructField("display", StringType())
        ])))
    ]))
])

# 2. Extract & Flatten
df_procedures = (
    df_bronze
    .withColumn("parsed", from_json(col("raw_payload"), procedure_schema))
    .filter(col("parsed.resourceType") == "Procedure")
    .select(
        col("parsed.id").alias("procedure_id"),
        regexp_replace(col("parsed.subject.reference"), "^urn:uuid:", "").alias("patient_id"),
        regexp_replace(col("parsed.encounter.reference"), "^urn:uuid:", "").alias("encounter_id"),
        col("parsed.code.coding")[0]["display"].alias("procedure_name"),
        col("parsed.code.coding")[0]["code"].alias("snomed_procedure_code"),
        col("parsed.status").alias("procedure_status"),
        to_timestamp(col("parsed.performedDateTime")).alias("performed_timestamp"),
        current_timestamp().alias("processed_at")
    )
)

# 3. Stream Write
query_procedures = (
    df_procedures.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/healthcare_catalog/default/checkpoints/chk_silver_procedure3")
    .trigger(availableNow=True)
    .toTable("healthcare_catalog.default.silver_procedures_staging")
)

In [0]:
%sql
MERGE INTO healthcare_catalog.default.silver_procedures t
USING (
    SELECT *
    FROM healthcare_catalog.default.silver_procedures_staging
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY procedure_id 
        ORDER BY processed_at DESC
    ) = 1
) s
ON t.procedure_id = s.procedure_id

WHEN MATCHED THEN
  UPDATE SET 
    t.patient_id = s.patient_id,
    t.encounter_id = s.encounter_id,
    t.procedure_name = s.procedure_name,
    t.snomed_procedure_code = s.snomed_procedure_code,
    t.procedure_status = s.procedure_status,
    t.performed_timestamp = s.performed_timestamp,
    t.processed_at = s.processed_at

WHEN NOT MATCHED THEN
  INSERT (
    procedure_id,
    patient_id,
    encounter_id,
    procedure_name,
    snomed_procedure_code,
    procedure_status,
    performed_timestamp,
    processed_at
  )
  VALUES (
    s.procedure_id,
    s.patient_id,
    s.encounter_id,
    s.procedure_name,
    s.snomed_procedure_code,
    s.procedure_status,
    s.performed_timestamp,
    s.processed_at
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
386,386,0,0
